# MSD MedSAM3 — resume the same 108 runs with adaptive concurrency

**No augmentation. Folds 0, 1, 3; seeds 17, 83, 191. Original 200 updates per run,
accumulation 2, native loss, medical pretrained LoRA, 10% target / 50% calibration p95 cap.**
This is the same study, not an additional 108 runs. Bands and bands+edge are still excluded.
The 27 historical results remain reference only. This study is update matched, not FLOP capped.

The scientific bundle is byte-for-byte unchanged. This notebook keeps the exact existing
**results_matched200_noaug_108_v1** folder, completed archives, calibration, preflight and checkpoints.
Each worker runs the original engine in a separate process with its own task, RNG and optimizer.
Only scheduling changes. Production CUDA numerical variability remains; concurrency is not a
guarantee of bitwise agreement with serial execution.

**Before switching:** interrupt the old Run cell once and wait until its clean-stop upload
finishes. Keep this VM connected. Do not delete the runtime or its local work folder.
Then open this notebook on that same runtime and run the cells below in order. If opening a
notebook gives you a different VM, first ensure the old cell has finished stopping: recovery
will then come from Drive. Use only one VM against this results folder.

Upload only **medsam3_msd_108_parallel_code.zip** to the existing `input` folder.
Keep the existing `medsam3_msd_data.zip` and `sam3.pt`; no new dataset or checkpoint upload is needed.


## 1. Connect to the same Drive results


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time
NOTEBOOK_STARTED = time.monotonic()
HOME_DIR = Path('/content/drive/MyDrive/hippo_medsam3_msd_extension')
INPUT = HOME_DIR/'input'
RESULTS = HOME_DIR/'results_matched200_noaug_108_v1'
BUNDLE = Path('/content/hippo_medsam3_msd_108_bundle')
CONTROLLER = Path('/content/hippo_medsam3_parallel_v1')
RESULTS.mkdir(parents=True, exist_ok=True)
print('Preserving and resuming:', RESULTS)


## 2. Verify and stage without replacing work
The existing scheduler lock must be free. Unchanged files are reused; differing files cause
a stop instead of replacement. Existing `work/`, recovery, archives and source bindings stay
intact. On a new VM only, this restores code and data from the existing uploads.


In [ ]:
def stage_parallel(package, data_zip, base_file, bundle, controller, results,
                   package_sha, manifest_sha, data_sha, base_sha):
    import fcntl
    import hashlib
    import json
    from pathlib import Path
    import shutil
    import zipfile

    package, data_zip, base_file, bundle, controller, results = map(
        Path, (package, data_zip, base_file, bundle, controller, results))

    def sha(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(8*1024**2), b''):
                h.update(block)
        return h.hexdigest()

    def safe(name):
        p = Path(name)
        if p.is_absolute() or '..' in p.parts:
            raise ValueError('Unsafe package path')
        return p

    def install_bytes(target, value, expected):
        if hashlib.sha256(value).hexdigest() != expected:
            raise ValueError('Package member hash mismatch: '+str(target))
        if target.is_symlink():
            raise ValueError('Refusing to overwrite symlink: '+str(target))
        if target.exists():
            if sha(target) != expected:
                raise ValueError('Existing file differs; preserved without replacement: '+str(target))
            return
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_name(target.name+'.staging')
        temp.write_bytes(value)
        temp.replace(target)

    if sha(package) != package_sha:
        raise ValueError('Upload hash mismatch: '+package.name)
    (bundle/'work').mkdir(parents=True, exist_ok=True)
    with (bundle/'work/sweep.lock').open('a') as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('A sweep is active. Stop the old Run cell once and wait for its recovery upload before staging.') from None
        existing = bundle/'bundle_manifest.json'
        if existing.exists() and sha(existing) != manifest_sha:
            raise ValueError('Existing bundle identity differs. No work has been replaced.')
        with zipfile.ZipFile(package) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)):
                raise ValueError('Duplicate package members')
            for info in archive.infolist():
                safe(info.filename)
                if (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Symlink in package')
            package_manifest = json.loads(archive.read('package_manifest.json'))
            if set(names) != set(package_manifest)|{'package_manifest.json'}:
                raise ValueError('Package file list differs')
            payload = archive.read('bundle/bundle_manifest.json')
            if hashlib.sha256(payload).hexdigest() != manifest_sha:
                raise ValueError('Scientific manifest differs')
            # Verify every existing scientific file before writing anything.
            for name, expected in package_manifest.items():
                part = safe(name)
                if part.parts[0] not in ('bundle', 'controller'):
                    raise ValueError('Unexpected package tree')
                target = (bundle if part.parts[0] == 'bundle' else controller)/Path(*part.parts[1:])
                if target.exists() and (target.is_symlink() or sha(target) != expected):
                    raise ValueError('Existing file differs; preserved: '+str(target))
            for name, expected in package_manifest.items():
                part = safe(name)
                target = (bundle if part.parts[0] == 'bundle' else controller)/Path(*part.parts[1:])
                install_bytes(target, archive.read(name), expected)
        data_manifest = json.loads((bundle/'data_manifest.json').read_text())
        missing = [name for name in data_manifest if not (bundle/'data'/safe(name)).exists()]
        for name, expected in data_manifest.items():
            path = bundle/'data'/safe(name)
            if path.exists() and sha(path) != expected:
                raise ValueError('Existing data changed: '+name)
        if missing:
            if sha(data_zip) != data_sha:
                raise ValueError('Data ZIP upload differs')
            with zipfile.ZipFile(data_zip) as archive:
                for name in missing:
                    install_bytes(bundle/'data'/safe(name), archive.read('data/'+name), data_manifest[name])
        base = bundle/'weights/sam3.pt'
        if base.exists():
            if sha(base) != base_sha:
                raise ValueError('Existing base checkpoint differs; preserved.')
        else:
            if shutil.disk_usage(bundle).free < 12*1024**3:
                raise RuntimeError('Need 12 GiB runtime disk space to stage the base checkpoint.')
            base.parent.mkdir(parents=True, exist_ok=True)
            temp = base.with_name('sam3.pt.staging')
            shutil.copyfile(base_file, temp)
            if sha(temp) != base_sha:
                raise ValueError('Base checkpoint copy differs')
            temp.replace(base)
        results.mkdir(parents=True, exist_ok=True)
        (results/'parallel_package.json').write_text(json.dumps(dict(
            package_sha256=package_sha, original_manifest_sha256=manifest_sha,
            science_source_changed=False, existing_results_preserved=True), indent=2))
    print('Original bundle and work preserved. Parallel supervisor staged separately.', flush=True)


stage_parallel(INPUT/'medsam3_msd_108_parallel_code.zip', INPUT/'medsam3_msd_data.zip', INPUT/'sam3.pt',
               BUNDLE, CONTROLLER, RESULTS, '776d2519a2b581d98dbb51d89714230ad52e714747171ec06b808bd7df216af0', '716f2e774fb9de3140baf1077eabff75fc0dc144f9a9e6752ec620dc5ddd6ac2',
               'ad869cf8c8862697b41bda740e91cae91bea6fcfdadca364b6f3ec5e6acf95e4', '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e')


## 3. Reuse the original environment and define notebook commands
An existing matching environment is reused without installation. A new VM installs the same
frozen requirements. Changing GPU/software after scientific work is blocked by the original
environment binding. Select the same A100 runtime type and High RAM.


In [ ]:
VENV = Path('/content/hippo_medsam3_venv')
PYTHON = VENV/'bin/python'
def setup(args):
    result = subprocess.run(list(map(str,args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with (RESULTS/'parallel_setup.log').open('a') as log: log.write(result.stdout)
    print('\n'.join(result.stdout.splitlines()[-35:]))
    if result.returncode: raise RuntimeError('Environment setup failed; see parallel_setup.log')
if not PYTHON.exists():
    setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
probe = subprocess.run([str(PYTHON),'-m','pip','--version'], capture_output=True, text=True)
if probe.returncode:
    import urllib.request
    bootstrap = Path('/content/medsam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', bootstrap)
    setup([PYTHON,bootstrap])
dependency_probe = """
import importlib.metadata as m, pathlib, sys
for line in pathlib.Path(sys.argv[1]).read_text().splitlines():
    if not line.strip(): continue
    parts = line.strip().split('==')
    try: version = m.version(parts[0])
    except m.PackageNotFoundError: sys.exit(1)
    if len(parts) == 2 and version != parts[1]: sys.exit(1)
"""
ready = subprocess.run([str(PYTHON),'-c',dependency_probe,str(BUNDLE/'requirements-colab.txt')]).returncode == 0
if not ready:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
else:
    print('Reusing the existing dependency versions.')
def execute(args, log_name, parallel=False):
    process = subprocess.Popen(list(map(str,args)), start_new_session=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    interrupted = False
    with (RESULTS/log_name).open('a') as log:
        while True:
            try:
                for line in process.stdout:
                    print(line,end='',flush=True); log.write(line); log.flush()
                code = process.wait()
                break
            except KeyboardInterrupt:
                interrupted = True
                print('Clean stop requested. Keep this VM connected until every worker verifies its Drive recovery.')
                if process.poll() is None:
                    if parallel: process.send_signal(signal.SIGUSR1)
                    else: os.killpg(process.pid, signal.SIGINT)
    if code: raise RuntimeError(f'Command failed. See {RESULTS/log_name}. Existing work is retained.')
    return not interrupted
def campaign(command, *extra):
    return execute([PYTHON,BUNDLE/'runtime.py',command,'--bundle',BUNDLE,'--durable',RESULTS,*extra], command+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## 4. Reuse the passed preflight
The original command verifies the same bundle and environment. A matching passed preflight
returns immediately without repeating calibration or GPU diagnostics. An incomplete preflight
resumes only its missing work. Scientific runs cannot bypass this check.


In [ ]:
campaign('preflight')


## 5. Run or resume with automatic worker selection
`MAX_WORKERS = 0` means automatic; it does **not** mean two workers. The supervisor starts with
one pending experiment, measures progress, then tries additional workers within GPU, CPU, RAM
and storage reserves. On an 80 GB A100 it can consider more than two workers; the actual useful
number is measured on this runtime. No extra benchmark experiments are created.

It compares stable training/evaluation throughput windows and stops increasing when a measured
gain is too small. Mixed phases use GPU activity and headroom to decide whether to try another
worker. Startup is staggered; reduced limits drain running work instead of discarding updates.
Resource estimates are conservative, not a guarantee against OOM. Memory pressure stops workers
cleanly and records a lower ceiling for the next resume. A failed update is retained for recovery
audit instead of replayed silently.

Saved unfinished baselines resume first; all nine baselines complete before new constraint runs.
Every 30 seconds, output shows **GPU usage, worker count, update/200 and evaluated volumes**.
Full logs stay in `BUNDLE/work/<experiment>/train.log`. Status and tuning decisions are also
written to Drive (`parallel_status.json`, `parallel_sessions/`).

Interrupt this cell **once** to stop all workers. Wait until it returns: local checkpoints are
saved every update, periodic Drive recovery continues, and a clean stop verifies every final
recovery upload. Partly completed evaluation and logs are also saved on a clean stop. Completed
archives are verified and skipped; completed bulky recovery files are retired by the original code.
A manual `RESULTS/STOP` is respected and never removed automatically.

The 22-hour setting includes time spent in this notebook's setup cells. It does not know time
spent in a previous notebook: lower it accordingly if this VM is already old. Colab can stop
earlier; a hard VM loss with newer charged updates than the saved recovery requires an audit.
Leave this VM running until the cell has confirmed clean stops; do not start another notebook
or VM writing the same results concurrently. On resume, run this cell again on the same VM;
on a new VM, run all setup cells first.


In [ ]:
MAX_WORKERS = 0  # Automatic: resource and throughput limited, not fixed at two.
MAX_NEW_RUNS = 108  # Upper bound on starts; already completed experiments are skipped.
SESSION_HOURS = 22.0
remaining_hours = SESSION_HOURS - (time.monotonic()-NOTEBOOK_STARTED)/3600
if remaining_hours <= 0.1:
    raise RuntimeError('Notebook session budget reached. Resume in a fresh runtime after clean shutdown.')
execute([PYTHON, CONTROLLER/'parallel.py', '--bundle', BUNDLE, '--durable', RESULTS,
         '--max-workers', str(MAX_WORKERS), '--max-runs', str(MAX_NEW_RUNS),
         '--hours', str(remaining_hours)], 'parallel_run.log', parallel=True)


## 6. Results with fresh matched controls
Same original comparison: validation Dice and gains against the fresh baseline for each fold
and seed. Historical bands/bands+edge runs remain reference only. No conditions are added.


In [ ]:
campaign('report')
import pandas as pd
comparison = json.loads((RESULTS/'comparison.json').read_text())
display(pd.DataFrame(comparison['aggregate']))
display(pd.DataFrame(comparison['runs']))
